# USGS Earthquake Data — Source Extraction

Fetches USGS earthquake events for 2026 to date into `sky.source.usgs__earthquake`, using a PySpark Custom Data Source that fetches date-range partitions in parallel — keeping every request under the API's 20,000-event and rate limits.

In [0]:
from datetime import datetime, timedelta, timezone

from pyspark.sql.datasource import DataSource, DataSourceReader, InputPartition
from pyspark.sql.types import MapType, StringType, StructField, StructType

# --- Configuration -----------------------------------------------------------
START_DATE = "2026-01-01"  # extract window start; the window always runs to now
NUM_PARTITIONS = 60  # ~5-day slices: under the 20,000-event cap per request
                     # and the 500-requests-per-5-minute rate limit
TARGET_TABLE = "sky.source.usgs__earthquake"

In [0]:
class TimeRangePartition(InputPartition):
    """One unit of work: a date range to fetch.

    Serialized and shipped to executors, so it stays lightweight:
    ISO strings for its slice of the window, plus an id for logging.
    """

    def __init__(self, starttime: str, endtime: str, pid: int):
        self.starttime = starttime
        self.endtime = endtime
        self.pid = pid

In [0]:
class USGSDataSourceReader(DataSourceReader):
    """Builds the partition list on the driver; fetches each partition on executors, in parallel."""

    def __init__(self, schema, options):
        self.schema = schema
        self.options = options

    @staticmethod
    def _parse_dt_utc(s: str) -> datetime:
        """Accepts 'YYYY-MM-DD' or ISO8601, normalized to UTC."""
        if len(s) == 10 and s[4] == "-" and s[7] == "-":
            return datetime.strptime(s, "%Y-%m-%d").replace(tzinfo=timezone.utc)
        dt = datetime.fromisoformat(s.replace("Z", "+00:00"))
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=timezone.utc)
        return dt.astimezone(timezone.utc)

    @staticmethod
    def _to_usgs_iso(dt: datetime) -> str:
        """Formats a datetime the way the USGS API expects."""
        return dt.strftime("%Y-%m-%dT%H:%M:%S")

    def partitions(self):
        """Runs on the DRIVER: split the requested window into even date-range slices."""
        start_s = self.options.get("starttime")
        end_s = self.options.get("endtime")
        if not start_s or not end_s:
            raise ValueError("Provide .option('starttime', ...) and .option('endtime', ...)")

        num_partitions = int(self.options.get("numPartitions", "10"))
        if num_partitions <= 0:
            raise ValueError("numPartitions must be > 0")

        start = self._parse_dt_utc(start_s)
        end = self._parse_dt_utc(end_s)
        if end <= start:
            raise ValueError(f"endtime must be after starttime. Got {start_s} -> {end_s}")

        step = (end - start).total_seconds() / num_partitions
        return [
            TimeRangePartition(
                starttime=self._to_usgs_iso(start + timedelta(seconds=step * i)),
                endtime=self._to_usgs_iso(start + timedelta(seconds=step * (i + 1))),
                pid=i,
            )
            for i in range(num_partitions)
        ]

    def read(self, partition: TimeRangePartition):
        """Runs on EXECUTORS: fetch one date range, yield rows matching the schema."""
        # Imported here — executors don't share the driver's namespace
        import requests

        response = requests.get(
            "https://earthquake.usgs.gov/fdsnws/event/1/query",
            params={
                "format": "geojson",
                "starttime": partition.starttime,
                "endtime": partition.endtime,
            },
            timeout=30,
        )
        if response.status_code != 200:
            raise RuntimeError(
                f"USGS HTTP {response.status_code} partition={partition.pid} "
                f"start={partition.starttime} end={partition.endtime}. "
                f"Response: {response.text[:500]}"
            )

        for feature in response.json().get("features", []):
            # Tuple matching the schema: (type, properties, geometry, id)
            yield (
                feature.get("type"),
                feature.get("properties"),
                feature.get("geometry"),
                feature.get("id"),
            )

In [0]:
class USGSDataSource(DataSource):
    """Entry point: registers under the name 'usgs' and defines the output schema."""

    @classmethod
    def name(cls) -> str:
        return "usgs"

    def schema(self) -> StructType:
        # properties and geometry stay as maps to preserve the full API payload
        return StructType([
            StructField("type", StringType(), True),
            StructField("properties", MapType(StringType(), StringType(), True), True),
            StructField("geometry", MapType(StringType(), StringType(), True), True),
            StructField("id", StringType(), True),
        ])

    def reader(self, schema: StructType) -> USGSDataSourceReader:
        return USGSDataSourceReader(schema, self.options)

In [0]:
spark.dataSource.register(USGSDataSource)
print("✓ USGS DataSource registered")

In [0]:
# Fetch the window in parallel, from START_DATE up to right now.
import time

end_now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
print(f"Fetching {START_DATE} through {end_now}")

start_time = time.time()

df = (spark.read.format("usgs")
    .option("starttime", START_DATE)
    .option("endtime", end_now)
    .option("numPartitions", str(NUM_PARTITIONS))
    .load()
    .cache())  # cache so the cells below don't re-hit the API

event_count = df.count()
print(f"Fetched {event_count:,} events in {time.time() - start_time:.2f} seconds")

In [0]:
# Keep the full raw API payload — silver extracts typed fields from these maps
from pyspark.sql.functions import col

df_expanded = df.select(
    col("id"),
    col("type"),
    col("properties"),  # all event details as a map
    col("geometry"),    # coordinates as a map
)

display(df_expanded.limit(10))

In [0]:
# Land the final DataFrame in the source layer
(df_expanded.write
    .mode("overwrite")
    .option("overwriteSchema", "true")  # allow the schema to evolve between runs
    .saveAsTable(TARGET_TABLE))

rows_written = spark.table(TARGET_TABLE).count()
print(f"✓ Saved {rows_written:,} rows to {TARGET_TABLE}")